# 05 - Define ECG-CXR CodeBind bridge

Frozen HuBERT-ECG and RAD-DINO features are projected into shared and modality-specific components. The shared components use one compositional codebook and are trained with symmetric same-row InfoNCE. Same-patient off-diagonal admissions are masked as negatives. No diagnostic labels or text participate.

In [ ]:
from pathlib import Path
import hashlib
import importlib
import os
import sys

import torch

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/path/to/ecg_cxr_bridge_artifacts",
))
CODE_ROOT = ARTIFACT_ROOT / "code"
CODE_ROOT.mkdir(parents=True, exist_ok=True)
module_path = CODE_ROOT / "ecg_cxr_codebind.py"


In [ ]:
module_source = r'''
from dataclasses import dataclass
import torch
from torch import nn
from torch.nn import functional as F

MODALITIES = ("ecg", "cxr")

class CompositionalCodebook(nn.Module):
    def __init__(self, size, code_dim, temperature=0.1):
        super().__init__()
        self.code_dim = int(code_dim)
        self.temperature = float(temperature)
        self.codes = nn.Parameter(torch.randn(size, code_dim))
        nn.init.normal_(self.codes, std=code_dim ** -0.5)

    def forward(self, features):
        if features.shape[-1] % self.code_dim:
            raise ValueError("Feature dimension must be divisible by code_dim")
        chunks = features.reshape(features.shape[0], -1, self.code_dim)
        logits = torch.einsum("bnd,kd->bnk", F.normalize(chunks, dim=-1), F.normalize(self.codes, dim=-1))
        indices = logits.argmax(-1)
        hard = F.embedding(indices, self.codes)
        commitment = F.mse_loss(chunks, hard.detach()) + F.mse_loss(hard, chunks.detach())
        quantized = chunks + (hard - chunks).detach()
        probabilities = F.softmax(logits / self.temperature, dim=-1).mean(1)
        return {"quantized": quantized.flatten(1), "indices": indices,
                "probabilities": probabilities, "commitment": commitment}

class Projector(nn.Module):
    def __init__(self, input_dim, common_dim, specific_dim, dropout):
        super().__init__()
        self.common_dim = int(common_dim)
        self.network = nn.Sequential(nn.LayerNorm(input_dim), nn.Dropout(dropout),
                                     nn.Linear(input_dim, common_dim + specific_dim))

    def forward(self, features):
        values = self.network(features)
        return values[:, :self.common_dim], values[:, self.common_dim:]

@dataclass
class ModalityOutput:
    common: torch.Tensor
    specific: torch.Tensor
    common_probabilities: torch.Tensor
    common_indices: torch.Tensor
    specific_indices: torch.Tensor
    commitment: torch.Tensor
    continuous_common: torch.Tensor
    continuous_specific: torch.Tensor

class ECGCXRCodeBind(nn.Module):
    def __init__(self, input_dims, common_dim=256, specific_dim=256, code_dim=8,
                 common_codebook_size=512, specific_codebook_size=256,
                 code_temperature=0.1, dropout=0.1):
        super().__init__()
        if set(input_dims) != set(MODALITIES):
            raise ValueError(f"input_dims must contain exactly {MODALITIES}")
        if common_dim % code_dim or specific_dim % code_dim:
            raise ValueError("common_dim and specific_dim must be divisible by code_dim")
        self.input_dims = dict(input_dims)
        self.projectors = nn.ModuleDict({name: Projector(input_dims[name], common_dim, specific_dim, dropout)
                                         for name in MODALITIES})
        self.common_codebook = CompositionalCodebook(common_codebook_size, code_dim, code_temperature)
        self.specific_codebooks = nn.ModuleDict({name: CompositionalCodebook(
            specific_codebook_size, code_dim, code_temperature) for name in MODALITIES})

    def encode_modality(self, name, features):
        continuous_common, continuous_specific = self.projectors[name](features)
        common = self.common_codebook(continuous_common)
        specific = self.specific_codebooks[name](continuous_specific)
        return ModalityOutput(F.normalize(common["quantized"], dim=-1),
                              F.normalize(specific["quantized"], dim=-1),
                              common["probabilities"], common["indices"], specific["indices"],
                              common["commitment"] + specific["commitment"],
                              continuous_common, continuous_specific)

    def forward(self, batch):
        return {name: self.encode_modality(name, batch[name]) for name in MODALITIES}

def symmetric_info_nce(left, right, subject_id, temperature=0.07):
    if len(left) < 2:
        return left.sum() * 0.0
    logits = left @ right.T / temperature
    same_patient = subject_id[:, None].eq(subject_id[None, :])
    off_diagonal = ~torch.eye(len(subject_id), dtype=torch.bool, device=left.device)
    logits = logits.masked_fill(same_patient & off_diagonal, -torch.inf)
    target = torch.arange(len(left), device=left.device)
    return 0.5 * (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target))

def jensen_shannon(left, right):
    eps = torch.finfo(left.dtype).eps
    midpoint = 0.5 * (left + right)
    return 0.5 * ((left * ((left + eps).log() - (midpoint + eps).log())).sum(-1)
                  + (right * ((right + eps).log() - (midpoint + eps).log())).sum(-1)).mean()

def orthogonality(output):
    shared = F.normalize(output.continuous_common, dim=0)
    specific = F.normalize(output.continuous_specific, dim=0)
    return (shared.T @ specific / max(1, len(shared))).square().mean()

def negative_entropy(probabilities):
    mean_probability = probabilities.mean(0)
    eps = torch.finfo(mean_probability.dtype).eps
    return (mean_probability * (mean_probability + eps).log()).sum()

def bridge_loss(outputs, batch, weights=None):
    weights = weights or {}
    alignment = symmetric_info_nce(outputs["ecg"].common, outputs["cxr"].common,
                                    batch["subject_id"], float(weights.get("temperature", 0.07)))
    code_relation = jensen_shannon(outputs["ecg"].common_probabilities,
                                   outputs["cxr"].common_probabilities)
    commitment = torch.stack([outputs[name].commitment for name in MODALITIES]).mean()
    orth = torch.stack([orthogonality(outputs[name]) for name in MODALITIES]).mean()
    usage = torch.stack([negative_entropy(outputs[name].common_probabilities) for name in MODALITIES]).mean()
    total = (float(weights.get("alignment_weight", 1.0)) * alignment
             + float(weights.get("code_relation_weight", 0.10)) * code_relation
             + float(weights.get("commitment_weight", 0.25)) * commitment
             + float(weights.get("orthogonality_weight", 0.05)) * orth
             + float(weights.get("code_usage_weight", 0.01)) * usage)
    return total, {"alignment": alignment, "code_relation": code_relation,
                   "commitment": commitment, "orthogonality": orth,
                   "code_usage": usage, "total": total}
'''
module_path.write_text(module_source)
print("Wrote:", module_path)
print("SHA256:", hashlib.sha256(module_source.encode()).hexdigest())


In [ ]:
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
import ecg_cxr_codebind
importlib.reload(ecg_cxr_codebind)
dims = {"ecg": 32, "cxr": 48}
model = ecg_cxr_codebind.ECGCXRCodeBind(
    dims, common_dim=16, specific_dim=16, code_dim=4,
    common_codebook_size=16, specific_codebook_size=8, dropout=0.0)
batch = {"ecg": torch.randn(8, 32), "cxr": torch.randn(8, 48),
         "subject_id": torch.arange(8)}
loss, components = ecg_cxr_codebind.bridge_loss(model(batch), batch)
assert torch.isfinite(loss)
loss.backward()
assert model.common_codebook.codes.grad is not None
print({name: float(value.detach()) for name, value in components.items()})
print("Forward/backward smoke test passed.")
